# Session XI: Python I Foundations Review

Reviews the **core concepts** from Sessions I–IX, the same building blocks you combined in the Session X Budget Manager lab.

---

## Exercise 1: Type Conversion & Error Handling

Write `safe_convert(values)` that takes a mixed list, tries `float()` on each, returns `(valid_list, error_count)`.

### How to approach this code

The input list mixes numbers, convertible strings, invalid strings, and `None`, so the function must continue after an individual conversion error. Catch `ValueError` and `TypeError`, keep valid floats, and count failures separately. Python also converts `True` to `1.0`; decide whether that is appropriate for this data.


In [1]:
def safe_convert(values: list) -> tuple[list[float], int]:
    """Convert each value with float(); keep the valid ones and count the failures."""
    # TODO
    valid = []
    error_count = 0

    for value in values:
        # float(True) is 1.0, but a bool is not a real number in this data, so count it as an error
        if isinstance(value, bool):
            error_count += 1
            continue
        try:
            valid.append(float(value))
        except (ValueError, TypeError):   # ValueError: "hello", "" / TypeError: None
            error_count += 1

    return valid, error_count

test = [10, "3.14", None, "hello", 0, "42", True, "", 7.5]
valid, errors = safe_convert(test)
print(f"Valid: {valid}")
print(f"Errors: {errors}")

Valid: [10.0, 3.14, 0.0, 42.0, 7.5]
Errors: 4


---
## Exercise 2: Data Structures & Comprehensions

Given students data, write:
- `get_honor_roll(students)` — names with avg >= 90
- `get_unique_courses(students)` — set of all courses
- `average_by_course(students)` — dict of course averages

### How to approach this code

Each student record contains a nested course-to-score dictionary. The honor roll is an ordered list, unique course names belong in a set, and average-by-course results fit a dictionary. The examples show how the question determines the data structure.


In [2]:
students = [
    {"name": "Alice", "courses": {"Math": 95, "Physics": 88, "CS": 92}},
    {"name": "Bob", "courses": {"Math": 78, "Chemistry": 85, "CS": 90}},
    {"name": "Carol", "courses": {"Physics": 92, "CS": 95, "Math": 91}},
    {"name": "David", "courses": {"Chemistry": 70, "Math": 65, "Biology": 80}},
    {"name": "Eve", "courses": {"CS": 98, "Math": 94, "Physics": 96}},
]

def get_honor_roll(students):
    """Return the names of the students whose average score is 90 or more."""
    # TODO
    honor_roll = []
    for student in students:
        scores = student["courses"].values()
        average = sum(scores) / len(scores)
        if average >= 90:
            honor_roll.append(student["name"])
    return honor_roll

def get_unique_courses(students):
    """Return a set with every course name."""
    # TODO
    return {course for student in students for course in student["courses"]}

def average_by_course(students):
    """Return a dict like {'Math': 84.6, 'CS': 93.75}."""
    # TODO
    totals = {}
    counts = {}
    for student in students:
        for course, score in student["courses"].items():
            totals[course] = totals.get(course, 0) + score
            counts[course] = counts.get(course, 0) + 1

    return {course: round(totals[course] / counts[course], 2) for course in totals}

print("Honor roll:", get_honor_roll(students))
print("Courses:", get_unique_courses(students))
print("Averages:", average_by_course(students))

Honor roll: ['Alice', 'Carol', 'Eve']
Courses: {'Biology', 'CS', 'Physics', 'Chemistry', 'Math'}
Averages: {'Math': 84.6, 'Physics': 92.0, 'CS': 93.75, 'Chemistry': 77.5, 'Biology': 80.0}


---
## Exercise 3: File I/O & String Cleaning

Parse messy CSV, clean names (title case), validate emails with regex, validate ages as int. Return `(valid_records, error_messages)`.

### How to approach this code

The CSV sample contains extra whitespace, inconsistent capitalization, a bad age, and malformed records. Clean each field before validating it; collect rejected rows with messages instead of letting one bad row stop the entire import. Regex tests email shape but cannot prove deliverability.


In [3]:
import csv
import io
import re

messy = """name,email,age
   alice SMITH , ALICE@EMAIL.COM , 25
 bob Jones,bob@email.com,thirty
  CAROL  white  , carol@email ,  30
 David Brown, david.brown@university.edu, 22
  , missing@email.com, 28
eve GREEN,eve.green@company.co.uk,27
"""

def clean_and_validate(raw: str) -> tuple[list[dict], list[str]]:
    """Clean each row, then keep the valid records and collect the error messages."""
    # TODO
    valid_records = []
    error_messages = []

    reader = csv.reader(io.StringIO(raw))
    next(reader)   # skip the header

    for line_number, row in enumerate(reader, 2):
        if len(row) != 3:
            error_messages.append(f"Line {line_number}: expected 3 fields, got {len(row)}")
            continue

        name = re.sub(r"\s+", " ", row[0]).strip().title()
        email = row[1].strip().lower()
        age_text = row[2].strip()

        if not name:
            error_messages.append(f"Line {line_number}: name is missing")
            continue
        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email):
            error_messages.append(f"Line {line_number}: invalid email {email!r}")
            continue
        try:
            age = int(age_text)
        except ValueError:
            error_messages.append(f"Line {line_number}: invalid age {age_text!r}")
            continue

        valid_records.append({"name": name, "email": email, "age": age})

    return valid_records, error_messages

valid, errors = clean_and_validate(messy)
print(f"Valid ({len(valid)}): {valid}")
print(f"Errors ({len(errors)}): {errors}")

Valid (3): [{'name': 'Alice Smith', 'email': 'alice@email.com', 'age': 25}, {'name': 'David Brown', 'email': 'david.brown@university.edu', 'age': 22}, {'name': 'Eve Green', 'email': 'eve.green@company.co.uk', 'age': 27}]
Errors (3): ["Line 3: invalid age 'thirty'", "Line 4: invalid email 'carol@email'", 'Line 6: name is missing']


---
## Exercise 4: OOP & Patterns

Build `Book` + `Library` classes. Library needs: add_book, search(keyword), checkout(title), available_books property, __len__, summary().

### How to approach this code

`Book` models one title and its checkout state; `Library` manages a collection of books. Search should not mutate the collection, while checkout changes availability only for a matching available book. Properties and special methods make the domain objects easier to inspect and use.


In [4]:
class Book:
    # TODO: __init__(title, author, year, is_available=True), __repr__
    def __init__(self, title, author, year, is_available=True):
        self.title = title
        self.author = author
        self.year = year
        self.is_available = is_available

    def __repr__(self):
        status = "available" if self.is_available else "checked out"
        return f"{self.title} by {self.author} ({self.year}) - {status}"

class Library:
    # TODO: add_book, search, checkout, available_books, __len__, summary
    def __init__(self):
        self.books = []

    def add_book(self, book):
        self.books.append(book)

    def search(self, keyword):
        """Return the books whose title or author contains the keyword (case-insensitive)."""
        keyword = keyword.lower()
        return [book for book in self.books
                if keyword in book.title.lower() or keyword in book.author.lower()]

    def checkout(self, title):
        """Check out an available book by title. Return True if it worked."""
        for book in self.books:
            if book.title.lower() == title.lower() and book.is_available:
                book.is_available = False
                return True
        return False

    @property
    def available_books(self):
        return [book for book in self.books if book.is_available]

    def __len__(self):
        return len(self.books)

    def summary(self):
        print(f"Library: {len(self)} books, {len(self.available_books)} available")
        for book in self.books:
            print(f"  - {book}")

lib = Library()
lib.add_book(Book("Python Crash Course", "Eric Matthes", 2019))
lib.add_book(Book("Clean Code", "Robert Martin", 2008))
lib.add_book(Book("Python Cookbook", "David Beazley", 2013))
print(f"Total: {len(lib)}")
print(f"Search 'python': {lib.search('python')}")
lib.checkout("Clean Code")
lib.summary()

Total: 3
Search 'python': [Python Crash Course by Eric Matthes (2019) - available, Python Cookbook by David Beazley (2013) - available]
Library: 3 books, 2 available
  - Python Crash Course by Eric Matthes (2019) - available
  - Clean Code by Robert Martin (2008) - checked out
  - Python Cookbook by David Beazley (2013) - available


---
## Exercise 5: Integrated project

Build a Contact Manager with: Contact class (name, email, phone, category), validate_email/phone with regex, JSON save/load, search with comprehensions, report grouped by category.

### How to approach this code

The contact manager combines a data model, static validation methods, JSON serialization, search, and category reports. The object keeps related fields together, validation protects the input boundary, and conversion methods allow records to be saved and reconstructed. Validate before persisting so bad data does not become permanent.


In [5]:
import json, re

class Contact:
    VALID_CATEGORIES = ["Family", "Friend", "Work", "Other"]
    # TODO: __init__, to_dict, from_dict, __repr__
    def __init__(self, name, email, phone, category="Other"):
        if not name.strip():
            raise ValueError("Name cannot be empty")
        if not Contact.validate_email(email):
            raise ValueError(f"Invalid email: {email!r}")
        if not Contact.validate_phone(phone):
            raise ValueError(f"Invalid phone: {phone!r}")
        if category not in Contact.VALID_CATEGORIES:
            raise ValueError(f"{category!r} is not a valid category. Choose from: {', '.join(Contact.VALID_CATEGORIES)}")

        self.name = name.strip().title()
        self.email = email.strip().lower()
        self.phone = phone.strip()
        self.category = category

    def to_dict(self):
        return {"name": self.name, "email": self.email, "phone": self.phone, "category": self.category}

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data["email"], data["phone"], data["category"])

    def __repr__(self):
        return f"{self.name} <{self.email}> {self.phone} [{self.category}]"

    # TODO: @staticmethod validate_email, validate_phone
    @staticmethod
    def validate_email(email):
        return bool(re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email.strip()))

    @staticmethod
    def validate_phone(phone):
        # optional +, then 7-15 digits that may be separated by spaces or dashes
        digits = re.sub(r"[\s-]", "", phone.strip())
        return bool(re.fullmatch(r"\+?\d{7,15}", digits))

def save_contacts(contacts, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump([contact.to_dict() for contact in contacts], f, indent=2)

def load_contacts(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        return []
    return [Contact.from_dict(item) for item in data]

def search_contacts(contacts, keyword):  # TODO
    keyword = keyword.lower()
    return [contact for contact in contacts
            if keyword in contact.name.lower() or keyword in contact.email]

def report_by_category(contacts):  # TODO
    groups = {}
    for contact in contacts:
        groups.setdefault(contact.category, []).append(contact)

    print("=== CONTACTS BY CATEGORY ===")
    for category in Contact.VALID_CATEGORIES:
        if category in groups:
            print(f"{category} ({len(groups[category])}):")
            for contact in groups[category]:
                print(f"  - {contact}")

# Uncomment to test:
contacts = [
    Contact("Alice", "alice@email.com", "612345678", "Friend"),
    Contact("bob smith", "Bob@Work.com", "+34 699 111 222", "Work"),
    Contact("Carol", "carol@family.org", "655-444-333", "Family"),
    Contact("Dan", "dan@work.com", "611222333", "Work"),
]
report_by_category(contacts)

print("Search 'work':", search_contacts(contacts, "work"))

save_contacts(contacts, "/tmp/test_contacts.json")
loaded = load_contacts("/tmp/test_contacts.json")
print("Round-trip:", [c.to_dict() for c in loaded] == [c.to_dict() for c in contacts])

try:
    Contact("Eve", "wrong-email", "612345678", "Friend")
except ValueError as error:
    print("Rejected:", error)

=== CONTACTS BY CATEGORY ===
Family (1):
  - Carol <carol@family.org> 655-444-333 [Family]
Friend (1):
  - Alice <alice@email.com> 612345678 [Friend]
Work (2):
  - Bob Smith <bob@work.com> +34 699 111 222 [Work]
  - Dan <dan@work.com> 611222333 [Work]
Search 'work': [Bob Smith <bob@work.com> +34 699 111 222 [Work], Dan <dan@work.com> 611222333 [Work]]
Round-trip: True
Rejected: Invalid email: 'wrong-email'


---
## Exercise 6: Review & Debugging Mini Lab
**Objective:** Practice reading code, spotting weak points, and improving readability.

Create a function called `review_code_snippet(snippet)` that:
- counts how many `if` statements appear
- flags repeated code patterns using a simple keyword search
- returns a small report dictionary with `if_count`, `repeated_patterns`, and `status`

This is intentionally a review exercise. The main goal is to think like a code reviewer.

### How to approach this code

The review helper treats source code as text and reports simple signals such as repeated calls or conditionals. This is a teaching heuristic rather than a static analyzer: counting text can be fooled by comments and strings. The structured result can guide a human review but cannot certify code quality.


In [6]:
import re


def review_code_snippet(snippet: str) -> dict:
    """Review a short code string and report likely quality issues."""
    # TODO: count `if` statements, look for repeated patterns like "print(" and "input(",
    # and return a summary dictionary with keys: "if_count", "repeated_patterns", "status"
    lines = [line.strip() for line in snippet.splitlines()]

    # an `if` statement starts the line with "if" or "elif" (this skips "else" and inline `x if y else z`)
    if_count = 0
    for line in lines:
        if re.match(r"(el)?if\b", line):
            if_count += 1

    # a pattern is "repeated" when it shows up 2 or more times
    repeated_patterns = {}
    for pattern in ["print(", "input(", "for ", "while "]:
        count = snippet.count(pattern)
        if count >= 2:
            repeated_patterns[pattern] = count

    if if_count > 3 or len(repeated_patterns) > 1:
        status = "needs review"
    elif repeated_patterns:
        status = "minor issues"
    else:
        status = "ok"

    return {"if_count": if_count, "repeated_patterns": repeated_patterns, "status": status}


# Example:
report = review_code_snippet("if x > 0:\n    print(x)\nif x < 0:\n    print(-x)")
print(report)
print(review_code_snippet("total = sum(values)\nprint(total)"))

{'if_count': 2, 'repeated_patterns': {'print(': 2}, 'status': 'minor issues'}
{'if_count': 0, 'repeated_patterns': {}, 'status': 'ok'}
